# 📊 พยากรณ์แนวโน้มการขึ้นทะเบียนเกษตรกรและพื้นที่เกษตรกรรมรายอำเภอ (Agricultural Registration Forecasting - Imbalance Handling)
## คลาสเรียนรู้ Machine Learning แบบรองรับ Data Imbalance (ความเบ้รายพื้นที่) ด้วย Target Log-Transformation
### 📊 แสดงผลแผนภูมิด้วย Plotly (Interactive Chart) เพื่อความสวยงามระดับ Premium และรองรับภาษาไทย 100%

---

### 📋 วัตถุประสงค์
1. วิเคราะห์และแก้ไขปัญหาความต่างทางขนาดพื้นที่เกษตรกรรมรายอำเภอ (Scale Imbalance / Skewness) ในเพชรบูรณ์ (เช่น อำเภอหนองไผ่ มีพื้นที่กว่า 400,000 ไร่ ในขณะที่อำเภอเขาค้อ มีเพียง 20,000 ไร่)
2. ประยุกต์ใช้วิธี **Target Log-Transformation** ในการแปลงข้อมูลเป้าหมายให้อยู่ในรูป $y' = \log(y + 1)$ เพื่อปรับน้ำหนักการเรียนรู้ของโมเดลให้เท่าเทียมกันทุกขนาดพื้นที่
3. สร้างและเปรียบเทียบโมเดล **Linear Regression (Log scale)** และ **Gradient Boosting Regressor (Log scale)**
4. ทำนายอนาคต 2 ปีข้างหน้า (ปี 2569 - 2570) แบบวนซ้ำ (Recursive Forecasting) เพื่อวางแผนการจัดการงบประมาณเยียวยาภัยพิบัติล่วงหน้า

## 1. Setup & Import Libraries 📦

In [1]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ตั้งค่า renderer สำหรับ VS Code Jupyter Notebook ให้แสดงผลกราฟิกแบบตอบสนองได้
pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)")

✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)


## 2. Load and Clean Dataset 📁

In [2]:
# กำหนด path ของข้อมูลการขึ้นทะเบียนเกษตรกร
file_path = os.path.join("data", "การขึ้นทะเบียนเกษตรกร", "การขึ้นทะเบียนเกษตรกร.csv")

# โหลดข้อมูล
df = pd.read_csv(file_path, encoding='utf-8-sig')

# ทำความสะอาดคอลัมน์และข้อมูลข้อความ
df.columns = [col.strip() for col in df.columns]
df['อำเภอ'] = df['อำเภอ'].str.strip()

print(f"📊 โหลดข้อมูลการขึ้นทะเบียนสำเร็จ! จำนวนแถวทั้งหมด: {len(df)} แถว")
display(df.head(11))

📊 โหลดข้อมูลการขึ้นทะเบียนสำเร็จ! จำนวนแถวทั้งหมด: 88 แถว


,ปี,อำเภอ,จำนวนครัวเรือน,จำนวนแปลง,เนื้อที่(ไร่)
0,2561,เมืองเพชรบูรณ์,15248,44470,314322.39
1,2561,ชนแดน,8743,19170,282037.54
2,2561,หล่มสัก,14560,39290,210108.37
3,2561,หล่มเก่า,8410,23317,213199.92
4,2561,วิเชียรบุรี,8509,19068,219649.00
5,2561,ศรีเทพ,7737,17837,202869.88
6,2561,หนองไผ่,11686,36187,410938.67
7,2561,บึงสามพัน,5170,11734,163691.29
8,2561,น้ำหนาว,2188,7971,100572.53
9,2561,วังโป่ง,3518,9566,120613.81


## 3. Data Imbalance Analysis (ความเบ้เชิงพื้นที่) ⚖️
วิเคราะห์สัดส่วนขนาดพื้นที่เกษตรกรรมระหว่างอำเภอต่างๆ เพื่อแสดงให้เห็นระดับความเหลื่อมล้ำทางขนาดของข้อมูล

In [3]:
# คำนวณค่าเฉลี่ยพื้นที่เพาะปลูกย้อนหลังรายอำเภอ
df_dist_avg = df.groupby('อำเภอ')['เนื้อที่(ไร่)'].mean().reset_index(name='พื้นที่เฉลี่ย(ไร่)')
df_dist_avg = df_dist_avg.sort_values(by='พื้นที่เฉลี่ย(ไร่)', ascending=False)

# แสดงด้วยกราฟแท่ง
fig = px.bar(
    df_dist_avg,
    x='อำเภอ',
    y='พื้นที่เฉลี่ย(ไร่)',
    title='การเปรียบเทียบขนาดพื้นที่เกษตรกรรมลงทะเบียนเฉลี่ยรายอำเภอ (พบความไม่สมดุลทางสเกลข้อมูลชัดเจน)',
    color='พื้นที่เฉลี่ย(ไร่)',
    color_continuous_scale=['#9575CD', '#311B92'] # โทนสีม่วงเข้มอย่างชัดเจน
)
fig.update_layout(
    xaxis_title='อำเภอ',
    yaxis_title='พื้นที่เฉลี่ย (ไร่)',
    coloraxis_showscale=False,
    title_x=0.5,
    height=500
)
fig.show()

## 4. Feature Engineering & Time-Series Preparation 🛠️
สร้างดัชนีเวลา และ Lag Features

In [4]:
# เรียงลำดับข้อมูลตามอำเภอและปี เพื่อสร้าง Lag Features
df_prep = df.sort_values(by=['อำเภอ', 'ปี']).reset_index(drop=True)

# สร้าง Lag 1 ปี
df_prep['lag_1_area'] = df_prep.groupby('อำเภอ')['เนื้อที่(ไร่)'].shift(1)
df_prep['lag_1_households'] = df_prep.groupby('อำเภอ')['จำนวนครัวเรือน'].shift(1)
df_prep['lag_1_plots'] = df_prep.groupby('อำเภอ')['จำนวนแปลง'].shift(1)

# สร้างดัชนีปี
df_prep['year_idx'] = df_prep['ปี'] - 2561

# ลบแถวปีแรก (2561) ที่ไม่มี Lag ย้อนหลัง
df_model = df_prep.dropna().reset_index(drop=True)

# ทำ One-Hot Encoding สำหรับอำเภอ
df_model = pd.get_dummies(df_model, columns=['อำเภอ'], prefix='dist', drop_first=False)

print("📊 ข้อมูลสำหรับการป้อนโมเดล:")
display(df_model.head(5))

📊 ข้อมูลสำหรับการป้อนโมเดล:


,ปี,จำนวนครัวเรือน,จำนวนแปลง,เนื้อที่(ไร่),lag_1_area,lag_1_households,lag_1_plots,year_idx,dist_ชนแดน,dist_น้ำหนาว,dist_บึงสามพัน,dist_วังโป่ง,dist_วิเชียรบุรี,dist_ศรีเทพ,dist_หนองไผ่,dist_หล่มสัก,dist_หล่มเก่า,dist_เขาค้อ,dist_เมืองเพชรบูรณ์
0,2562,10115,25226,315201.31,282037.54,8743.0,19170.0,1,True,False,False,False,False,False,False,False,False,False,False
1,2563,11208,30602,440556.20,315201.31,10115.0,25226.0,2,True,False,False,False,False,False,False,False,False,False,False
2,2564,10388,27579,387814.11,440556.20,11208.0,30602.0,3,True,False,False,False,False,False,False,False,False,False,False
3,2565,9391,24198,336926.60,387814.11,10388.0,27579.0,4,True,False,False,False,False,False,False,False,False,False,False
4,2566,9181,31086,430987.65,336926.60,9391.0,24198.0,5,True,False,False,False,False,False,False,False,False,False,False


## 5. Train-Test Split & Target Log-Transformation 🤖
ทำการแปลงข้อมูลคำตอบ (Target) ในกลุ่ม Train ให้อยู่บนสเกลลอการิทึมด้วย `np.log1p` เพื่อบีบระยะห่างความไม่สมดุล

In [5]:
# คัดเลือก Features และ Target
dist_cols = [col for col in df_model.columns if col.startswith('dist_')]
features = ['year_idx', 'lag_1_area', 'lag_1_households', 'lag_1_plots'] + dist_cols

target_area = 'เนื้อที่(ไร่)'
target_households = 'จำนวนครัวเรือน'
target_plots = 'จำนวนแปลง'

# แบ่งกลุ่มข้อมูล Train / Test ตามแกนเวลา
train_mask = df_model['ปี'] < 2568
test_mask = df_model['ปี'] == 2568

df_train = df_model[train_mask]
df_test = df_model[test_mask]

X_train, X_test = df_train[features], df_test[features]

# ทำการแปลงเป้าหมายในชุด Train ให้เป็น Log Scale ด้วย np.log1p
y_train_area_log = np.log1p(df_train[target_area])
y_train_hh_log = np.log1p(df_train[target_households])
y_train_plots_log = np.log1p(df_train[target_plots])

y_test_area = df_test[target_area]
y_test_hh = df_test[target_households]
y_test_plots = df_test[target_plots]

# 1. โมเดลพยากรณ์พื้นที่การเกษตร (Area Forecast)
lr_area = LinearRegression().fit(X_train, y_train_area_log)
gbr_area = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_area_log)

# 2. โมเดลพยากรณ์จำนวนครัวเรือน (Households Forecast)
lr_hh = LinearRegression().fit(X_train, y_train_hh_log)
gbr_hh = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_hh_log)

# 3. โมเดลพยากรณ์จำนวนแปลง (Plots Forecast)
lr_plots = LinearRegression().fit(X_train, y_train_plots_log)
gbr_plots = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_plots_log)

# ประเมินประสิทธิภาพผลลัพธ์ปี 2568 (โดยแปลงกลับสเกลเดิมด้วย np.expm1)
lr_pred_area = np.expm1(lr_area.predict(X_test))
gbr_pred_area = np.expm1(gbr_area.predict(X_test))

print(f"\n{'='*20} 🎯 ประสิทธิภาพพื้นที่ปี 2568 (วัดบนสเกลจริงหลังจากแปลงกลับ) {'='*20}")
print(f"[Linear Regression (Log)] R2: {r2_score(y_test_area, lr_pred_area):.4f} | MAE: {mean_absolute_error(y_test_area, lr_pred_area):,.2f}")
print(f"[Gradient Boosting (Log)] R2: {r2_score(y_test_area, gbr_pred_area):.4f} | MAE: {mean_absolute_error(y_test_area, gbr_pred_area):,.2f}")


==================== 🎯 ประสิทธิภาพพื้นที่ปี 2568 (วัดบนสเกลจริงหลังจากแปลงกลับ) ====================
[Linear Regression (Log)] R2: 0.6702 | MAE: 46,320.49
[Gradient Boosting (Log)] R2: 0.7633 | MAE: 30,823.40


## 6. Recursive Future Forecasting (ปี 2569 - 2570) 🔮
ทำนายแบบวนซ้ำรายอำเภอ โดยโมเดลจะทำนายค่าเป็น Log จากนั้นแปลงกลับสเกลจริงเพื่อนำไปใช้เป็น Lag Feature ของลูปเวลาถัดไป

In [6]:
districts = df['อำเภอ'].unique()
forecast_results = []

for dist in districts:
    dist_latest = df_prep[(df_prep['อำเภอ'] == dist) & (df_prep['ปี'] == 2568)].iloc[0]
    
    current_area = dist_latest['เนื้อที่(ไร่)']
    current_hh = dist_latest['จำนวนครัวเรือน']
    current_plots = dist_latest['จำนวนแปลง']
    
    dist_onehot = {col: 1 if col == f'dist_{dist}' else 0 for col in dist_cols}
    
    for year in [2569, 2570]:
        year_idx = year - 2561
        
        input_data = {
            'year_idx': year_idx,
            'lag_1_area': current_area,
            'lag_1_households': current_hh,
            'lag_1_plots': current_plots
        }
        input_data.update(dist_onehot)
        
        X_input = pd.DataFrame([input_data])[features]
        
        # โมเดลทำนายค่าออกมาเป็น Log scale
        pred_area_log = gbr_area.predict(X_input)[0]
        pred_hh_log = gbr_hh.predict(X_input)[0]
        pred_plots_log = gbr_plots.predict(X_input)[0]
        
        # แปลงกลับเป็นสเกลจริง
        pred_area = np.expm1(pred_area_log)
        pred_hh = np.expm1(pred_hh_log)
        pred_plots = np.expm1(pred_plots_log)
        
        forecast_results.append({
            'ปี': year,
            'อำเภอ': dist,
            'เนื้อที่(ไร่)': max(0, pred_area),
            'จำนวนครัวเรือน': max(0, int(pred_hh)),
            'จำนวนแปลง': max(0, int(pred_plots)),
            'ประเภท': 'พยากรณ์ (Forecast)'
        })
        
        # กำหนดค่ากลับสำหรับปีถัดไป
        current_area = pred_area
        current_hh = pred_hh
        current_plots = pred_plots

df_forecast = pd.DataFrame(forecast_results)
print("🔮 คำทำนายปี 2569 และ 2570 (หลังปรับแก้ข้อมูลไม่สมดุลแล้ว):")
display(df_forecast.head(10))

🔮 คำทำนายปี 2569 และ 2570 (หลังปรับแก้ข้อมูลไม่สมดุลแล้ว):


,ปี,อำเภอ,เนื้อที่(ไร่),จำนวนครัวเรือน,จำนวนแปลง,ประเภท
0,2569,เมืองเพชรบูรณ์,323439.294520,16856,48005,พยากรณ์ (Forecast)
1,2570,เมืองเพชรบูรณ์,326189.215590,16508,47909,พยากรณ์ (Forecast)
2,2569,ชนแดน,247527.577194,8896,21519,พยากรณ์ (Forecast)
3,2570,ชนแดน,204488.844953,8885,21595,พยากรณ์ (Forecast)
4,2569,หล่มสัก,174742.872660,14086,34993,พยากรณ์ (Forecast)
5,2570,หล่มสัก,173381.536427,13135,35929,พยากรณ์ (Forecast)
6,2569,หล่มเก่า,169476.567440,8884,24635,พยากรณ์ (Forecast)
7,2570,หล่มเก่า,169476.567440,8963,24358,พยากรณ์ (Forecast)
8,2569,วิเชียรบุรี,380929.208356,13560,38179,พยากรณ์ (Forecast)
9,2570,วิเชียรบุรี,380929.208356,13187,37631,พยากรณ์ (Forecast)


## 7. Visualization 📊
รวมผลข้อมูลและจำลองอนาคตที่สะท้อนการปรับระดับสเกลข้อมูลเรียบร้อยแล้ว

In [7]:
df_historical = df[['ปี', 'อำเภอ', 'เนื้อที่(ไร่)', 'จำนวนครัวเรือน', 'จำนวนแปลง']].copy()
df_historical['ประเภท'] = 'ข้อมูลจริง (Actual)'
df_combined = pd.concat([df_historical, df_forecast], ignore_index=True)

# แสดงผลพยากรณ์พื้นที่การเกษตร
fig = px.line(
    df_combined,
    x='ปี',
    y='เนื้อที่(ไร่)',
    color='อำเภอ',
    line_dash='ประเภท',
    markers=True,
    title='พยากรณ์พื้นที่ทำการเกษตรรายอำเภอ จังหวัดเพชรบูรณ์ (ปี 2561 - 2570) [โมเดลแก้ปัญหาข้อมูลเบ้]'
)
fig.update_layout(
    xaxis_title='ปีงบประมาณ',
    yaxis_title='พื้นที่ลงทะเบียน (ไร่)',
    title_x=0.5,
    height=600,
    hovermode='x unified'
)
fig.show()

In [8]:
# แสดงผลพยากรณ์จำนวนครัวเรือน
fig_hh = px.line(
    df_combined,
    x='ปี',
    y='จำนวนครัวเรือน',
    color='อำเภอ',
    line_dash='ประเภท',
    markers=True,
    title='พยากรณ์จำนวนครัวเรือนเกษตรกรรายอำเภอ จังหวัดเพชรบูรณ์ (ปี 2561 - 2570) [โมเดลแก้ปัญหาข้อมูลเบ้]'
)
fig_hh.update_layout(
    xaxis_title='ปีงบประมาณ',
    yaxis_title='จำนวนครัวเรือน (ครัวเรือน)',
    title_x=0.5,
    height=600,
    hovermode='x unified'
)
fig_hh.show()

## 8. Feature Importance of GBR (Log) Model 🏅

In [9]:
importances = gbr_area.feature_importances_
indices = np.argsort(importances)[::-1]

imp_df = pd.DataFrame({
    'Feature': [features[i] for i in indices],
    'Importance': importances[indices]
}).head(8)

fig_imp = px.bar(
    imp_df.sort_values('Importance', ascending=True),
    y='Feature',
    x='Importance',
    orientation='h',
    title='ค่าน้ำหนักความสำคัญของตัวแปรในโมเดลพยากรณ์พื้นที่การเกษตร (GBR - Log Scale)',
    color='Importance',
    color_continuous_scale=['#9575CD', '#311B92']
)
fig_imp.update_layout(
    yaxis_title='ตัวแปร/ปัจจัย',
    xaxis_title='ค่าน้ำหนักความสำคัญ (Relative Importance)',
    coloraxis_showscale=False,
    height=400,
    title_x=0.5
)
fig_imp.show()